# Baseline — Sentence-BERT 嵌入 + LogisticRegression 烟雾测试版

跟 VADER 烟雾版同款结构：默认只跑前 N 个块，确认整条管道（编码 / 时间安全切分 / 训练 / 指标）在本机或服务器跑通，之后再放大到全量。

默认 `SMOKE=True`，N = 10。MLP 头为了保持最小化先去掉，全量版再补回来；输出写到 `outputs/sentencebert_smoke/`。

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DEFAULT_INPUT_PATH, TRAIN_RATIO, VAL_RATIO, TEST_RATIO  # noqa: E402
from data import build_comment_blocks, load_posts, temporal_split_blocks  # noqa: E402
from debate_graph.text_embeddings import encode_texts  # noqa: E402

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_INPUT_PATH =", DEFAULT_INPUT_PATH)

In [ ]:
# 一行开关：True = 仅前 N 个块烟雾测试；False = 跑全量
SMOKE = False
LIMIT_BLOCKS = 10 if SMOKE else None
BACKEND = "sentencebert"

posts = load_posts(DEFAULT_INPUT_PATH)
blocks, issues = build_comment_blocks(posts)
if LIMIT_BLOCKS is not None:
    blocks = blocks[:LIMIT_BLOCKS]
print(f"blocks={len(blocks)} issues={len(issues)} smoke={SMOKE} backend={BACKEND}")

def build_block_text(block):
    parts = [
        f"POST: {(block.post_content or '').strip()[:2000]}",
        f"ROOT_COMMENT ({block.root_comment.author}): {(block.root_comment.text or '').strip()[:2000]}",
    ]
    return "\n\n".join(parts)


texts = [build_block_text(block) for block in blocks]
embeddings = encode_texts(texts, backend=BACKEND)
print(f"embedding_dim={len(embeddings[0]) if embeddings else 0}")

# 时间安全切分（烟雾版只用 train/test）
split = temporal_split_blocks(blocks, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO)
id_to_index = {id(block): idx for idx, block in enumerate(blocks)}


def collect(piece):
    X, y = [], []
    for block in piece:
        X.append(embeddings[id_to_index[id(block)]])
        y.append(int(block.label))
    return X, y


X_train, y_train = collect(split.train)
X_test, y_test = collect(split.test)
print(f"train={len(X_train)} test={len(X_test)} (skip val to keep smoke minimal)")

import numpy as np

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)

# 1/-1 → 0/1
label_to_class = {1: 0, -1: 1}
class_to_label = {0: 1, 1: -1}
y_train_cls = np.asarray([label_to_class[int(v)] for v in y_train])
y_test_cls = np.asarray([label_to_class[int(v)] for v in y_test])
y_test_int = np.asarray(y_test, dtype=np.int64)

In [ ]:
from sklearn.linear_model import LogisticRegression

LABEL_BULLISH, LABEL_BEARISH = 1, -1
lr = LogisticRegression(max_iter=2000, C=1.0)
lr.fit(X_train_np, y_train_cls)
pred_cls = lr.predict(X_test_np)
pred_int = np.asarray([class_to_label[int(v)] for v in pred_cls], dtype=np.int64)

pairs = list(zip(y_test_int.tolist(), pred_int.tolist()))
n = len(pairs)
correct = sum(1 for t, p in pairs if t == p)
acc = correct / n if n else 0.0
matrix = {"bullish": {"bullish": 0, "bearish": 0}, "bearish": {"bullish": 0, "bearish": 0}}
for t, p in pairs:
    matrix["bullish" if t == 1 else "bearish"]["bullish" if p == 1 else "bearish"] += 1
print(f"LR test n={n} acc={acc:.3f}")
print(f"confusion_matrix={matrix}")

In [ ]:
OUT_DIR = PROJECT_ROOT / "outputs" / "sentencebert"
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = [
    {
        "block_id": split.test[i].block_id,
        "t0": split.test[i].t0.strftime("%Y-%m-%d %H:%M:%S"),
        "true_label": int(y_test_int[i]),
        "pred_label_int": int(pred_int[i]),
    }
    for i in range(n)
]
summary = {
    "config": {
        "input": str(DEFAULT_INPUT_PATH),
        "limit_blocks": LIMIT_BLOCKS,
        "smoke": SMOKE,
        "embedding_backend": BACKEND,
        "splits": {"train": len(split.train), "val": len(split.val), "test": len(split.test)},
    },
    "total": n,
    "accuracy": acc,
    "confusion_matrix": matrix,
    "predictions": rows,
}
(OUT_DIR / "metrics.json").write_text(
    json.dumps({k: v for k, v in summary.items() if k != "predictions"}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
with (OUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as fp:
    for row in rows:
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")
print("wrote", OUT_DIR / "metrics.json")
print("wrote", OUT_DIR / "predictions.jsonl")

## 如何切到全量

烟雾版跑通后，把第二个代码 cell 顶部的开关改成：

```python
SMOKE = False
LIMIT_BLOCKS = None  # 全量
```

再把输出目录（`OUT_DIR`）改成 `outputs/sentencebert/`，整本 Run All 即可。

服务器上无交互跑：

```bash
"D:/anaconda/envs/sentiment/python.exe" -m jupyter nbconvert --to notebook --execute \
    --ExecutePreprocessor.timeout=3600 \
    notebooks/02_sentencebert_baseline.ipynb \
    --output 02_sentencebert_baseline_full.ipynb
```

备注：

- 烟雾版只用了 LogReg 分类头；全量版如果想加 MLP，参考原 `02_sentencebert_baseline.ipynb` 草稿（已挪到 git 历史）。
- Sentence-BERT 权重会缓存到 `outputs/hf_cache/`，只在第一次时联网下载。